Selic acumulada

In [3]:
import pandas as pd
import sqlite3
from pathlib import Path

# Caminho do CSV baixado
csv_path = Path.cwd().parent / "data" / "raw" / "bcdata.sgs.4390.csv"

# Ler CSV (formato do BCB: separador ; e decimal ,)
df_selic_mensal = pd.read_csv(csv_path, sep=";", decimal=",")
df_selic_mensal["data"] = pd.to_datetime(df_selic_mensal["data"], dayfirst=True)
df_selic_mensal["ano"] = df_selic_mensal["data"].dt.year

# Filtrar apenas 2021 a 2025
df_selic_mensal = df_selic_mensal[df_selic_mensal["ano"].between(2021, 2025)]

# Converter % mensal em fator e compor (juros compostos)
df_selic_mensal["fator"] = 1 + (df_selic_mensal["valor"] / 100)
selic_acumulada_anual = df_selic_mensal.groupby("ano")["fator"].prod() - 1

# Transformar em DataFrame e converter para %
selic_acumulada_anual = selic_acumulada_anual.reset_index()
selic_acumulada_anual.rename(columns={"fator":"selic_acumulada"}, inplace=True)
selic_acumulada_anual["selic_acumulada"] *= 100

print(selic_acumulada_anual)

# Salvar no SQLite
db_path = Path.cwd().parent / "data" / "nubank.db"
conn = sqlite3.connect(db_path)
selic_acumulada_anual.to_sql("selic_anual", conn, if_exists="replace", index=False)
conn.close()

    ano  selic_acumulada
0  2021         4.435688
1  2022        12.380487
2  2023        13.028231
3  2024        10.888126
4  2025        14.332756
